# Airbnb Exploratory Data Analysis
## Paris & Lyon


## 01 · Data sources and setup


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

VOLUME = "/Volumes/airbnb_lakehouse/bronze/raw_files"
CITY_PATHS = {
    "Paris": f"{VOLUME}/paris_listings.csv.gz",
    "Lyon": f"{VOLUME}/lyon_listings.csv.gz",
}
SNAPSHOT_DATES = {"Paris": "2026-09-16", "Lyon": "2026-09-17"}


def load_listings(path):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(path)
    )

raw = {city: load_listings(path) for city, path in CITY_PATHS.items()}
paris, lyon = raw["Paris"], raw["Lyon"]

for city, df in raw.items():
    print(f"{city} | snapshot: {SNAPSHOT_DATES[city]} | rows: {df.count():,} | columns: {len(df.columns)}")


## 02 · Dataset integrity and schema


In [0]:
paris_cols, lyon_cols = set(paris.columns), set(lyon.columns)
print("Shared columns:", len(paris_cols & lyon_cols))
print("Only in Paris:", sorted(paris_cols - lyon_cols))
print("Only in Lyon:", sorted(lyon_cols - paris_cols))

key_columns = [
    "id", "price", "room_type", "accommodates", "bedrooms", "beds",
    "bathrooms_text", "neighbourhood_cleansed", "availability_365",
    "number_of_reviews", "review_scores_rating"
]

raw_types = [
    (city, column, dict(df.dtypes).get(column, "missing"))
    for city, df in raw.items()
    for column in key_columns
]
display(spark.createDataFrame(raw_types, ["city", "column", "raw_type"]).orderBy("column", "city"))

for city, df in raw.items():
    print(f"{city} — source data preview")
    display(df.select(*key_columns).limit(5))


In [0]:
for city, df in raw.items():
    total = df.count()
    unique_ids = df.select("id").distinct().count()
    print(f"{city}: {total:,} rows | {unique_ids:,} distinct IDs | {total - unique_ids:,} duplicate IDs")


## 03 · Missing values


In [0]:
def missing_report(df, city):
    n = df.count()
    expressions = [
        F.sum(
            F.when(
                F.col(name).isNull() | (F.trim(F.col(name)) == ""), 1
            ).otherwise(0)
        ).alias(name)
        for name in df.columns
    ]
    counts = df.agg(*expressions).first().asDict()
    rows = [
        (city, name, int(counts[name]), round(100 * counts[name] / n, 2))
        for name in df.columns
    ]
    return spark.createDataFrame(rows, ["city", "column", "missing_count", "missing_pct"])

missing_summary = missing_report(paris, "Paris").unionByName(missing_report(lyon, "Lyon"))
missing_comparison = (
    missing_summary.groupBy("column")
    .pivot("city", ["Paris", "Lyon"])
    .agg(F.first("missing_pct"))
    .orderBy(F.desc("Paris"), "column")
)

display(missing_comparison)


In [0]:
FEATURES = [
    "room_type", "accommodates", "bedrooms", "beds", "bathrooms_text",
    "neighbourhood_cleansed", "latitude", "longitude", "minimum_nights"
]

focus_columns = FEATURES + ["price", "availability_365", "review_scores_rating"]
display(
    missing_comparison
    .filter(F.col("column").isin(focus_columns))
    .orderBy("column")
)


## 04 · Reusable typed features


In [0]:
def parse_price(column):
    return F.expr(f"try_cast(regexp_replace(`{column}`, '[$€,]', '') AS DOUBLE)")


def prepare_listings(df, city):
    text = F.lower(F.trim(F.col("bathrooms_text")))
    number = F.regexp_extract(text, r"(\d+(?:\.\d+)?)", 1)

    return (
        df.withColumn("city", F.lit(city))
        .withColumn("price_numeric", parse_price("price"))
        .withColumn("quoted_price_numeric", parse_price("price_quote_price_per_night"))
        .withColumn(
            "has_valid_price",
            F.col("price_numeric").isNotNull() & (F.col("price_numeric") > 0),
        )
        .withColumn("availability_numeric", F.expr("try_cast(availability_365 AS INT)"))
        .withColumn(
            "bathrooms_numeric",
            F.when(text.rlike(r"half[- ]bath"), F.lit(0.5))
            .when(number != "", number.cast("double"))
            .otherwise(F.lit(None).cast("double")),
        )
        .withColumn(
            "bathroom_type",
            F.when(text.isNull(), "Unknown")
            .when(text.contains("shared"), "Shared")
            .when(text.contains("private"), "Private")
            .otherwise("Standard"),
        )
        .withColumn(
            "bathroom_anomaly",
            (F.col("bathrooms_numeric") > 8)
            & ((F.col("room_type") == "Private room") | (F.expr("try_cast(accommodates AS INT)") <= 5)),
        )
    )

prepared = {city: prepare_listings(df, city) for city, df in raw.items()}
paris_eda, lyon_eda = prepared["Paris"], prepared["Lyon"]
combined_eda = paris_eda.unionByName(lyon_eda)

print("Derived EDA column types:")
for name in ["price_numeric", "has_valid_price", "availability_numeric", "bathrooms_numeric", "bathroom_type"]:
    print(f"  {name}: {dict(paris_eda.dtypes)[name]}")


## 05 · Price coverage and missingness patterns


In [0]:
price_coverage = (
    combined_eda.groupBy("city")
    .agg(
        F.count("*").alias("total_listings"),
        F.sum(F.col("has_valid_price").cast("int")).alias("valid_prices"),
    )
    .withColumn("missing_or_invalid", F.col("total_listings") - F.col("valid_prices"))
    .withColumn("valid_pct", F.round(100 * F.col("valid_prices") / F.col("total_listings"), 2))
    .orderBy("city")
)

display(price_coverage)

display(
    combined_eda.groupBy("city", "room_type")
    .agg(
        F.count("*").alias("listings"),
        F.sum((~F.col("has_valid_price")).cast("int")).alias("missing_prices"),
    )
    .withColumn("missing_pct", F.round(100 * F.col("missing_prices") / F.col("listings"), 2))
    .orderBy("city", F.desc("missing_pct"))
)


## 06 · Price distribution and high-value anomalies


In [0]:
price_stats = (
    combined_eda.filter(F.col("has_valid_price"))
    .groupBy("city")
    .agg(
        F.count("*").alias("listings"),
        F.round(F.avg("price_numeric"), 2).alias("mean"),
        F.round(F.min("price_numeric"), 2).alias("minimum"),
        F.round(F.percentile_approx("price_numeric", 0.25), 2).alias("q25"),
        F.round(F.percentile_approx("price_numeric", 0.50), 2).alias("median"),
        F.round(F.percentile_approx("price_numeric", 0.75), 2).alias("q75"),
        F.round(F.percentile_approx("price_numeric", 0.95), 2).alias("p95"),
        F.round(F.percentile_approx("price_numeric", 0.99), 2).alias("p99"),
        F.round(F.max("price_numeric"), 2).alias("maximum"),
    )
    .orderBy("city")
)

display(price_stats)

display(
    combined_eda.filter(F.col("has_valid_price"))
    .groupBy("city", "room_type")
    .agg(
        F.count("*").alias("listings"),
        F.round(F.percentile_approx("price_numeric", 0.5), 2).alias("median_price"),
    )
    .orderBy("city", "room_type")
)


In [0]:
for city, df in prepared.items():
    valid = df.filter(F.col("has_valid_price"))
    median, p99 = valid.approxQuantile("price_numeric", [0.5, 0.99], 0.001)
    prices = (
        valid.filter(F.col("price_numeric") <= p99)
        .select("price_numeric")
        .toPandas()["price_numeric"]
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.hist(prices, bins=55, edgecolor="white", linewidth=0.4)
    ax.axvline(median, linestyle="--", label=f"Median: {median:.2f}")
    ax.set(title=f"{city} | Advertised nightly price", xlabel="Source price units", ylabel="Listings")
    ax.grid(axis="y", alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)


In [0]:
outlier_rows = []

for city, df in prepared.items():
    valid = df.filter(F.col("has_valid_price"))
    q1, median, q3 = valid.approxQuantile("price_numeric", [0.25, 0.5, 0.75], 0.001)
    upper_fence = q3 + 1.5 * (q3 - q1)
    total = valid.count()
    outliers = valid.filter(F.col("price_numeric") > upper_fence).count()
    extreme = valid.filter(F.col("price_numeric") > 10 * median).count()
    outlier_rows.append((city, total, round(upper_fence, 2), outliers, round(100 * outliers / total, 2), extreme))

outlier_summary = spark.createDataFrame(
    outlier_rows,
    ["city", "priced_listings", "iqr_upper_fence", "above_fence", "outlier_pct", "above_10x_median"],
)
display(outlier_summary)


In [0]:
for city, df in prepared.items():
    print(f"{city} | 10 most expensive listings — review candidates, not confirmed errors")
    display(
        df.filter(F.col("has_valid_price"))
        .select("id", "name", "price_numeric", "room_type", "accommodates", "bedrooms", "neighbourhood_cleansed")
        .orderBy(F.desc("price_numeric"))
        .limit(10)
    )


## 07 · Calendar availability


In [0]:
availability_stats = (
    combined_eda.groupBy("city")
    .agg(
        F.count("*").alias("listings"),
        F.round(F.avg("availability_numeric"), 2).alias("mean_days"),
        F.percentile_approx("availability_numeric", 0.5).alias("median_days"),
        F.min("availability_numeric").alias("minimum"),
        F.max("availability_numeric").alias("maximum"),
        F.sum(
            (F.col("availability_numeric").isNull() | ~F.col("availability_numeric").between(0, 365)).cast("int")
        ).alias("invalid_availability"),
    )
    .orderBy("city")
)
display(availability_stats)

availability_df = (
    combined_eda.withColumn(
        "availability_group",
        F.when(F.col("availability_numeric") == 0, "0 days")
        .when(F.col("availability_numeric").between(1, 90), "1–90 days")
        .when(F.col("availability_numeric").between(91, 180), "91–180 days")
        .when(F.col("availability_numeric").between(181, 270), "181–270 days")
        .when(F.col("availability_numeric").between(271, 365), "271–365 days")
        .otherwise("Invalid / missing"),
    )
)

availability_distribution = (
    availability_df.groupBy("city", "availability_group")
    .agg(F.count("*").alias("listings"))
    .withColumn("total_city", F.sum("listings").over(Window.partitionBy("city")))
    .withColumn("percentage", F.round(100 * F.col("listings") / F.col("total_city"), 2))
    .orderBy("city", "availability_group")
)
display(availability_distribution)


In [0]:
buckets = ["0 days", "1–90 days", "91–180 days", "181–270 days", "271–365 days"]
chart_data = availability_distribution.select("city", "availability_group", "percentage").toPandas()

for city in CITY_PATHS:
    city_values = chart_data[chart_data["city"] == city].set_index("availability_group")
    values = [float(city_values.loc[b, "percentage"]) for b in buckets]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(buckets, values)
    ax.set(title=f"{city} | Availability in the next 365 days", xlabel="Available-day range", ylabel="Share of listings (%)")
    ax.set_ylim(0, max(values) * 1.2)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


In [0]:
price_missing_by_availability = (
    availability_df.groupBy("city", "availability_group")
    .agg(
        F.count("*").alias("listings"),
        F.sum((~F.col("has_valid_price")).cast("int")).alias("missing_prices"),
    )
    .withColumn("missing_price_pct", F.round(100 * F.col("missing_prices") / F.col("listings"), 2))
    .orderBy("city", "availability_group")
)
display(price_missing_by_availability)


## 08 · Feature readiness and bathroom engineering


In [0]:
feature_quality = (
    missing_summary.filter(F.col("column").isin(FEATURES))
    .groupBy("column")
    .pivot("city", ["Paris", "Lyon"])
    .agg(F.first("missing_pct"))
    .orderBy("column")
)
display(feature_quality)

numeric_rules = {"accommodates": 1, "bedrooms": 0, "beds": 0, "minimum_nights": 1}
numeric_rows = []

for city, df in prepared.items():
    priced = df.filter(F.col("has_valid_price"))
    for name, minimum in numeric_rules.items():
        number = F.expr(f"try_cast(`{name}` AS DOUBLE)")
        row = priced.agg(
            F.count("*").alias("total"),
            F.sum((number.isNull() | (number < minimum)).cast("int")).alias("invalid"),
            F.min(number).alias("min_value"),
            F.max(number).alias("max_value"),
        ).first()
        numeric_rows.append((city, name, row["total"], row["invalid"], row["min_value"], row["max_value"]))

numeric_quality = spark.createDataFrame(
    numeric_rows, ["city", "feature", "priced_listings", "missing_or_invalid", "min_value", "max_value"]
)
display(numeric_quality)


In [0]:
bathroom_quality = (
    combined_eda.groupBy("city")
    .agg(
        F.count("*").alias("listings"),
        F.sum(F.col("bathrooms_numeric").isNull().cast("int")).alias("unparsed"),
        F.sum((F.col("bathrooms_numeric") == 0).cast("int")).alias("reported_zero"),
        F.sum(F.col("bathroom_anomaly").cast("int")).alias("flagged_for_review"),
    )
    .withColumn("unparsed_pct", F.round(100 * F.col("unparsed") / F.col("listings"), 2))
    .orderBy("city")
)
display(bathroom_quality)

for city, df in prepared.items():
    print(f"{city} | unusual bathroom records (sample)")
    display(
        df.filter(F.col("bathroom_anomaly"))
        .select("id", "name", "room_type", "accommodates", "bedrooms", "bathrooms_text", "bathrooms_numeric")
        .orderBy(F.desc("bathrooms_numeric"))
        .limit(10)
    )


## 09 · Target leakage review


In [0]:
LEAKAGE_COLUMNS = [
    "price", "price_quote_checkin_date", "price_quote_checkout_date",
    "price_quote_total_price", "price_quote_price_per_night", "price_quote_raw",
    "estimated_revenue_l365d",
]

leakage_missingness = (
    missing_summary.filter(F.col("column").isin(LEAKAGE_COLUMNS))
    .groupBy("column")
    .pivot("city", ["Paris", "Lyon"])
    .agg(F.first("missing_pct"))
    .orderBy("column")
)
display(leakage_missingness)

leakage_check = (
    combined_eda.filter(
        F.col("price_numeric").isNotNull() & F.col("quoted_price_numeric").isNotNull()
    )
    .withColumn("absolute_difference", F.abs(F.col("price_numeric") - F.col("quoted_price_numeric")))
    .groupBy("city")
    .agg(
        F.count("*").alias("compared_listings"),
        F.round(100 * F.avg((F.col("absolute_difference") <= 0.01).cast("int")), 2).alias("exact_match_pct"),
        F.round(F.avg("absolute_difference"), 2).alias("mean_absolute_difference"),
    )
    .orderBy("city")
)
display(leakage_check)

print("Leakage-related columns excluded from model inputs:", LEAKAGE_COLUMNS)
print("Candidate features shared by both cities:", all(c in paris.columns and c in lyon.columns for c in FEATURES))
